# Data Cleaning - Maison_d'hôte Vente Marrakech
This notebook focuses on cleaning the maison_d'hôte vente data for Marrakech.

In [ ]:
# Conversion des prix en DH si nécessaire
if 'df' in locals():
    if 'prix' in df.columns and 'prix_num' in df.columns:
        # Taux de conversion approximatif (1 EUR = 10.8 MAD)
        EUR_TO_MAD = 10.8
        
        def convert_to_dh(row):
            price_str = str(row['prix']).lower() if pd.notna(row['prix']) else ''
            price_num = row['prix_num']
            
            # Si le prix contient l'euro et qu'on n'a pas encore une valeur en DH cohérente
            if '€' in price_str or 'eur' in price_str:
                import re
                # Extraire le montant en euro (souvent le premier nombre)
                nums = re.findall(r'\d+', price_str.replace(' ', '').replace(',', ''))
                if nums:
                    # Si c'est en euros, on multiplie par notre taux
                    euro_val = float(nums[0])
                    # On vérifie si price_num est déjà une conversion correcte (proche de euro_val * 10)
                    # Sinon on force la conversion
                    if pd.isna(price_num) or price_num < euro_val * 5:
                        return round(euro_val * EUR_TO_MAD)
            
            return price_num
            
        df['prix_num'] = df.apply(convert_to_dh, axis=1)
        print("Prix uniformisés en DH.")

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# File path
file_path = '../../data/marrakech_immo_vente/maison_hote_vente.csv'

# Load data
if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Successfully loaded {file_path}")
    display(df.head())
else:
    print(f"ERROR: File not found at {file_path}")
    print(f"Current working directory: {os.getcwd()}")

Successfully loaded ../../data/marrakech_immo_vente/maison_hote_vente.csv


,id,titre,prix,localisation,type_bien,surface,chambres,salles_bain,description,agence,...,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,quartier,prix_m2,prix_m2_median_quartier
0,RR368,Exceptionnel Boutique-Riad de 8 suites avec em...,Sur demande Dhs,Marrakech,Maison d'Hôtes,NC m²,NaN,NaN,Ce bien est situé à proximité immédiate du Pal...,Marrakech Realty,...,-1,NaN,NaN,NaN,0.0,0.0,1.0,Autre,NaN,23103.448276
1,VMK323,Vaste écolodge à vendre à proximité de Marrake...,Upon request Dhs,Marrakech,Maison d'Hôtes,m²,NaN,NaN,Il s'agit d'une propriété rare sur le marché p...,Marrakech Realty,...,-1,NaN,NaN,NaN,0.0,0.0,1.0,Autre,NaN,23103.448276
2,RR400,Exceptionnel Palais d’Hôtes idéalement situé :...,Upon request Dhs,Marrakech,Maison d'Hôtes,m²,NaN,NaN,"The Riad aux allures de Palais, de 1 200 m2 au...",Marrakech Realty,...,-1,NaN,NaN,NaN,0.0,0.0,1.0,Autre,NaN,23103.448276
3,VMK382,Exceptionnel Ecolodge à rénover à proximité de...,Upon Request Dhs,Marrakech,Maison d'Hôtes,NC m²,NaN,NaN,Ce bien unique est une superbe opportunité pou...,Marrakech Realty,...,-1,NaN,NaN,NaN,0.0,0.0,1.0,Autre,NaN,23103.448276
4,VMK423,Elégant Boutique-Hôtel de 11 suites en Palmeraie,SUR DEMANDE Dhs,Marrakech,Maison d'Hôtes,900 m²,NaN,NaN,Ce bien de prestige est idéalement situé dans ...,Marrakech Realty,...,-1,NaN,NaN,900.0,0.0,0.0,1.0,Autre,NaN,23103.448276


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       71 non-null     str    
 1   titre                    71 non-null     str    
 2   prix                     71 non-null     str    
 3   localisation             71 non-null     str    
 4   type_bien                71 non-null     str    
 5   surface                  71 non-null     str    
 6   chambres                 49 non-null     float64
 7   salles_bain              0 non-null      float64
 8   description              70 non-null     str    
 9   agence                   71 non-null     str    
 10  url                      71 non-null     str    
 11  source                   71 non-null     str    
 12  piscine                  71 non-null     int64  
 13  parking                  71 non-null     int64  
 14  ascenseur                71 non-null   

In [3]:
df.describe()

,chambres,salles_bain,piscine,parking,ascenseur,terrasse,jardin,climatisation,securite,vue,...,hammam,etage,surface_terrain,prix_num,surface_num,chambres_num,salles_bain_num,nb_pieces,prix_m2,prix_m2_median_quartier
count,49.000000,0.0,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,71.000000,...,71.000000,71.000000,0.0,6.500000e+01,40.00000,71.000000,71.0,71.000000,39.000000,7.100000e+01
mean,8.816327,NaN,0.760563,0.183099,0.042254,0.830986,0.267606,0.112676,0.084507,0.253521,...,0.492958,-0.563380,NaN,1.403654e+07,681.67500,6.084507,0.0,7.042254,23665.932758,2.310345e+04
std,4.381179,NaN,0.429777,0.389500,0.202599,0.377432,0.445862,0.318447,0.280126,0.438123,...,0.503509,0.578974,NaN,9.246410e+06,423.82673,5.479172,0.0,5.506974,7705.193855,7.327744e-12
min,4.000000,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-1.000000,NaN,3.850000e+06,160.00000,0.000000,0.0,1.000000,4500.000000,2.310345e+04
25%,6.000000,NaN,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-1.000000,NaN,7.685000e+06,372.50000,0.000000,0.0,1.000000,18438.644689,2.310345e+04
50%,7.000000,NaN,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-1.000000,NaN,1.038000e+07,575.00000,6.000000,0.0,7.000000,23103.448276,2.310345e+04
75%,11.000000,NaN,1.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.500000,...,1.000000,0.000000,NaN,1.855000e+07,970.00000,8.000000,0.0,9.000000,30312.668464,2.310345e+04
max,26.000000,NaN,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,2.000000,NaN,4.840000e+07,2200.00000,26.000000,0.0,27.000000,38750.000000,2.310345e+04


In [4]:
df.columns

Index(['id', 'titre', 'prix', 'localisation', 'type_bien', 'surface',
       'chambres', 'salles_bain', 'description', 'agence', 'url', 'source',
       'piscine', 'parking', 'ascenseur', 'terrasse', 'jardin',
       'climatisation', 'securite', 'vue', 'meuble', 'neuf', 'cave', 'hammam',
       'etage', 'surface_terrain', 'prix_num', 'surface_num', 'chambres_num',
       'salles_bain_num', 'nb_pieces', 'quartier', 'prix_m2',
       'prix_m2_median_quartier'],
      dtype='str')

## 1. Initial Data Overview

In [6]:
if 'df' in locals():
    print("Shape:", df.shape)
    df.info()
else:
    print("DataFrame 'df' not loaded. Please fix the file path above.")

Shape: (71, 34)
<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 34 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       71 non-null     str    
 1   titre                    71 non-null     str    
 2   prix                     71 non-null     str    
 3   localisation             71 non-null     str    
 4   type_bien                71 non-null     str    
 5   surface                  71 non-null     str    
 6   chambres                 49 non-null     float64
 7   salles_bain              0 non-null      float64
 8   description              70 non-null     str    
 9   agence                   71 non-null     str    
 10  url                      71 non-null     str    
 11  source                   71 non-null     str    
 12  piscine                  71 non-null     int64  
 13  parking                  71 non-null     int64  
 14  ascenseur              

## 2. Handling Missing Values

## 3. Removing Duplicates

In [7]:
if 'df' in locals():
    initial_size = len(df)
    df = df.drop_duplicates()
    print(f"Removed {initial_size - len(df)} duplicate rows.")

Removed 0 duplicate rows.


## 4. Cleaning Numerical Columns
Handling outliers for price and surface.

In [8]:
if 'df' in locals():
    if 'prix_num' in df.columns:
        q_low = df['prix_num'].quantile(0.01)
        q_hi  = df['prix_num'].quantile(0.99)
        df = df[(df['prix_num'] <= q_hi) & (df['prix_num'] >= q_low)]
        print(f"Data points after price outlier removal: {len(df)}")
    
    if 'surface_num' in df.columns:
        q_low_surf = df['surface_num'].quantile(0.01)
        q_hi_surf  = df['surface_num'].quantile(0.99)
        df = df[(df['surface_num'] <= q_hi_surf) & (df['surface_num'] >= q_low_surf)]
        print(f"Data points after surface outlier removal: {len(df)}")

Data points after price outlier removal: 63
Data points after surface outlier removal: 36


## 5. Saving Cleaned Data

In [9]:
if 'df' in locals():
    output_dir = '../../data/cleaned_data/vente'
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    output_file = 'maison_hote_vente_cleaned.csv'
    output_path = os.path.join(output_dir, output_file)
    df.to_csv(output_path, index=False)
    print(f"Cleaned data saved to {output_path}")

Cleaned data saved to ../../data/cleaned_data/vente/maison_hote_vente_cleaned.csv
